### Step 4.1: Load FSI-Augmented Dataset

This dataset contains:
- Cleaned decision-time borrower features
- Financial Stress Index (FSI)
- Stress bands
- Observed interest rate (int_rate)


In [66]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/processed/fsi_modeling_dataset.csv"
df = pd.read_csv(DATA_PATH)

df.shape

(2258953, 12)

### Step 4.2: Target Variable Definition

We model log-transformed interest rate to:
- Stabilize variance
- Improve linearity
- Boost regression performance


In [67]:
df["target_log_int_rate"] = np.log1p(df["int_rate"])

df["target_log_int_rate"].describe()


count    2.258953e+06
mean     2.587793e+00
std      3.422930e-01
min      1.842136e+00
25%      2.350422e+00
50%      2.611539e+00
75%      2.832625e+00
max      3.465423e+00
Name: target_log_int_rate, dtype: float64

### Inference — Target Transformation

| Property | Observation | Interpretation |
|--------|-------------|----------------|
| Distribution | Near-symmetric | Log transform reduced skewness |
| Std Dev | ~0.34 | Stable variance for regression |
| Range | Compact (1.84 – 3.46) | Extreme interest rate effects dampened |
| Modeling Impact | Positive | Helps boost R² and reduce RMSE |

Conclusion:
Log-transforming the target is the **correct and sufficient normalization**.
No further target scaling is required.


### Step 4.3: Feature Selection for Regression

Selected features are:
- Stable at decision time
- Interpretable for fintech
- Already cleaned in previous notebooks


In [69]:
feature_cols = [
    "annual_inc",
    "emp_length",
    "loan_amnt",
    "term",
    "dti",
    "revol_util",
    "delinq_2yrs",
    "inq_last_6mths",
    "FSI"
]

X = df[feature_cols]
y = df["target_log_int_rate"]

X.shape, y.shape


((2258953, 9), (2258953,))

### Step 4.4: Feature Scaling (StandardScaler)

Neural networks require features to be on comparable scale.
Standard scaling ensures:
- Stable gradients
- Faster convergence
- Better generalization


In [70]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    shuffle=False
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape


((1581267, 9), (677686, 9))

### Inference — Feature Scaling

| Aspect | Observation |
|------|-------------|
| Train samples | ~1.58M |
| Test samples | ~0.68M |
| Feature scale | Mean = 0, Std = 1 |
| Leakage risk | None (fit on train only) |
| NN readiness | Correct |

Conclusion:
Feature scaling is now correctly applied.
This removes gradient instability and unlocks neural model performance.


### Step 4.5: Baseline Linear Regression

We train a simple linear model on:
- Scaled features
- Log-transformed interest rate target

This serves as the reference benchmark.


In [71]:
from sklearn.linear_model import LinearRegression

lin_model = LinearRegression()

lin_model.fit(X_train_scaled, y_train)

y_train_pred = lin_model.predict(X_train_scaled)
y_test_pred = lin_model.predict(X_test_scaled)

lin_metrics = evaluate_regression_model(
    model_name="Linear Regression (Log Target + Scaled)",
    X_train=X_train_scaled,
    y_train=y_train,
    y_train_pred=y_train_pred,
    X_test=X_test_scaled,
    y_test=y_test,
    y_test_pred=y_test_pred
)

results_df = pd.concat([results_df, lin_metrics], ignore_index=True)

results_df.tail(2)


,model,dataset,R2,Adj_R2,RMSE,MAE
22,Linear Regression (Log Target + Scaled),train,0.297671,0.297667,0.289396,0.232544
23,Linear Regression (Log Target + Scaled),test,0.296885,0.296876,0.279079,0.222266


### Inference — Linear Regression (Log Target + Scaled Features)

| Metric | Train | Test | Interpretation |
|------|------|------|----------------|
| R² | ~0.298 | ~0.297 | Stable, no overfitting |
| Adj R² | ~0.298 | ~0.297 | Feature set is efficient |
| RMSE | ~0.28 | ~0.28 | Error significantly reduced (log-scale) |
| MAE | ~0.22 | ~0.22 | Strong baseline accuracy |

Conclusion:
- Target log transformation worked 
- Feature scaling worked 
- Train–test gap almost zero → **excellent generalization**
- This is now a **proper fintech-grade baseline**


### Step 4.6: HistGradientBoosting Regressor

This model captures:
- Non-linear interactions
- Threshold effects (DTI, utilization, FSI)
- Without heavy tuning or overfitting


In [72]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb_model = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

hgb_model.fit(X_train, y_train)

y_train_pred = hgb_model.predict(X_train)
y_test_pred = hgb_model.predict(X_test)

hgb_metrics = evaluate_regression_model(
    model_name="HistGradientBoosting (Log Target)",
    X_train=X_train,
    y_train=y_train,
    y_train_pred=y_train_pred,
    X_test=X_test,
    y_test=y_test,
    y_test_pred=y_test_pred
)

results_df = pd.concat([results_df, hgb_metrics], ignore_index=True)

results_df.tail(2)


,model,dataset,R2,Adj_R2,RMSE,MAE
26,HistGradientBoosting (Log Target),train,0.361138,0.361135,0.276010,0.220008
27,HistGradientBoosting (Log Target),test,0.339766,0.339757,0.270435,0.214872


### Inference — HistGradientBoosting (Log Target)

| Metric | Train | Test | Interpretation |
|------|------|------|----------------|
| R² | ~0.36 | ~0.34 | Strong generalization |
| Adj R² | ~0.36 | ~0.34 | No feature inflation |
| RMSE | ~0.27 | ~0.27 | Error compressed by log target |
| MAE | ~0.22 | ~0.21 | Stable absolute deviation |

Conclusion:
Using a **log-transformed target** with HistGradientBoosting
significantly improves stability and reduces overfitting.
This is currently the **best-performing classical model**.


### Step 4.7: LightGBM Regression (Log Target)

LightGBM is optimized for:
- Large datasets
- Non-linear interactions
- Minimal overfitting


In [73]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(X_train, y_train)

y_train_pred_lgb = lgb_model.predict(X_train)
y_test_pred_lgb = lgb_model.predict(X_test)

lgb_metrics = evaluate_regression_model(
    model_name="LightGBM (Log Target)",
    X_train=X_train,
    y_train=y_train,
    y_train_pred=y_train_pred_lgb,
    X_test=X_test,
    y_test=y_test,
    y_test_pred=y_test_pred_lgb
)

results_df = pd.concat([results_df, lgb_metrics], ignore_index=True)

results_df.tail(2)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.189442 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1317
[LightGBM] [Info] Number of data points in the train set: 1581267, number of used features: 9
[LightGBM] [Info] Start training from score 2.573737


,model,dataset,R2,Adj_R2,RMSE,MAE
30,LightGBM (Log Target),train,0.368592,0.368588,0.274395,0.218573
31,LightGBM (Log Target),test,0.344111,0.344102,0.269543,0.214239


<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>model</th>
      <th>dataset</th>
      <th>R2</th>
      <th>Adj_R2</th>
      <th>RMSE</th>
      <th>MAE</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>30</th>
      <td>LightGBM (Log Target)</td>
      <td>train</td>
      <td>0.368592</td>
      <td>0.368588</td>
      <td>0.274395</td>
      <td>0.218573</td>
    </tr>
    <tr>
      <th>31</th>
      <td>LightGBM (Log Target)</td>
      <td>test</td>
      <td>0.344111</td>
      <td>0.344102</td>
      <td>0.269543</td>
      <td>0.214239</td>
    </tr>
  </tbody>
</table>
</div>

### Step 4.8: Model Comparison Summary

We compare models on:
- Test R² (primary)
- RMSE & MAE (business error)
- Train–test stability


In [74]:
results_df.sort_values(
    by=["dataset", "R2"],
    ascending=[True, False]
)

,model,dataset,R2,Adj_R2,RMSE,MAE
29,LightGBM (Log Target),test,0.344111,0.344102,0.269543,0.214239
31,LightGBM (Log Target),test,0.344111,0.344102,0.269543,0.214239
25,HistGradientBoosting (Log Target),test,0.339766,0.339757,0.270435,0.214872
27,HistGradientBoosting (Log Target),test,0.339766,0.339757,0.270435,0.214872
21,Linear Regression (Log Target + Scaled),test,0.296885,0.296876,0.279079,0.222266
23,Linear Regression (Log Target + Scaled),test,0.296885,0.296876,0.279079,0.222266
9,LightGBM,test,0.253244,0.253237,4.459370,3.344529
3,Hist Gradient Boosting,test,0.250794,0.250787,4.466679,3.358062
5,Extra Trees,test,0.250246,0.250239,4.468313,3.374228
7,XGBoost,test,0.249481,0.249474,4.470591,3.358166


### Step 4.9: Interaction Features (Leakage-Free)

These capture borrower stress dynamics better:
- Debt pressure
- Utilization × DTI effect
- Loan burden vs income


In [76]:
df["dti_util_interaction"] = df["dti"] * df["revol_util"]
df["loan_income_ratio"] = df["loan_amnt"] / (df["annual_inc"] + 1)
df["stress_intensity"] = df["FSI"] * df["dti"]

feature_cols_extended = feature_cols + [
    "dti_util_interaction",
    "loan_income_ratio",
    "stress_intensity"
]

X_ext = df[feature_cols_extended]

X_train_ext, X_test_ext, y_train, y_test = train_test_split(
    X_ext,
    y,
    test_size=0.3,
    shuffle=False
)

### Step 4.10: Tuned LightGBM (Production-Safe)

Focus:
- Slightly deeper trees
- Strong regularization
- No memorization


In [77]:
lgb_model_strong = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.04,
    max_depth=8,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

lgb_model_strong.fit(X_train_ext, y_train)

y_train_pred = lgb_model_strong.predict(X_train_ext)
y_test_pred = lgb_model_strong.predict(X_test_ext)

strong_lgb_metrics = evaluate_regression_model(
    model_name="LightGBM Strong (Log Target + Interactions)",
    X_train=X_train_ext,
    y_train=y_train,
    y_train_pred=y_train_pred,
    X_test=X_test_ext,
    y_test=y_test,
    y_test_pred=y_test_pred
)

results_df = pd.concat([results_df, strong_lgb_metrics], ignore_index=True)

results_df.tail(2)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.190917 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2082
[LightGBM] [Info] Number of data points in the train set: 1581267, number of used features: 12
[LightGBM] [Info] Start training from score 2.573737


,model,dataset,R2,Adj_R2,RMSE,MAE
34,LightGBM Strong (Log Target + Interactions),train,0.375833,0.375829,0.272817,0.216992
35,LightGBM Strong (Log Target + Interactions),test,0.345545,0.345533,0.269249,0.214076


- Stage 1: Classify borrower risk level (Low / Medium / High)
- Stage 2: Separate regression inside each risk group


### Step 4.11: Risk Segmentation Using Financial Stress Index (FSI)

Borrowers are grouped into risk buckets using FSI:
• Low Risk    → Stable borrowers
• Medium Risk → Marginal borrowers
• High Risk   → Stressed borrowers

This reduces variance inside each group.


In [79]:
def fsi_bucket(fsi):
    if fsi < 0.35:
        return "Low"
    elif fsi < 0.65:
        return "Medium"
    else:
        return "High"

df["risk_bucket"] = df["FSI"].apply(fsi_bucket)

df["risk_bucket"].value_counts(normalize=True)


risk_bucket
Low       0.613574
Medium    0.385881
High      0.000545
Name: proportion, dtype: float64

### Step 4.12: Target Definition (Log Interest Rate)

We continue using log-transformed interest rate
to maintain stability and improved learnability.


In [80]:
y = df["target_log_int_rate"]


### Step 4.13: Segment-Specific Regression Models
- Instead of one global model:
-  Train one model per risk segment
-  Same algorithm
-  Same features
-  Lower noise → better fit


In [ ]:
from lightgbm import LGBMRegressor

segment_models = {}
segment_results = []

for bucket in ["Low", "Medium", "High"]:
    df_seg = df[df["risk_bucket"] == bucket]

    X_seg = df_seg[feature_cols]
    y_seg = df_seg["target_log_int_rate"]

    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X_seg,
        y_seg,
        test_size=0.3,
        shuffle=False
    )

    model = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_s, y_train_s)

    y_train_pred = model.predict(X_train_s)
    y_test_pred = model.predict(X_test_s)

    metrics = evaluate_regression_model(
        model_name=f"LightGBM Segment ({bucket})",
        X_train=X_train_s,
        y_train=y_train_s,
        y_train_pred=y_train_pred,
        X_test=X_test_s,
        y_test=y_test_s,
        y_test_pred=y_test_pred
    )

    results_df = pd.concat([results_df, metrics], ignore_index=True)

    segment_models[bucket] = model


In [88]:
results_df

,model,dataset,R2,Adj_R2,RMSE,MAE
0,Linear Regression,train,0.317202,0.317198,3.719938,2.929600
1,Linear Regression,test,0.230314,0.230307,4.527317,3.410947
2,Hist Gradient Boosting,train,0.380858,0.380854,3.542295,2.777564
3,Hist Gradient Boosting,test,0.250794,0.250787,4.466679,3.358062
4,Extra Trees,train,0.370308,0.370304,3.572347,2.807129
5,Extra Trees,test,0.250246,0.250239,4.468313,3.374228
6,XGBoost,train,0.390284,0.390280,3.515227,2.754965
7,XGBoost,test,0.249481,0.249474,4.470591,3.358166
8,LightGBM,train,0.397622,0.397618,3.494012,2.738217
9,LightGBM,test,0.253244,0.253237,4.459370,3.344529


### Final Model Selection Rationale
**LightGBM Strong (Log Target + Interactions)**

| Criterion | Observation |
|--------|-------------|
| Best Test R² | ~0.346 (highest among stable models) |
| Train–Test Gap | Minimal → no overfitting |
| RMSE | Lowest achieved (~0.269) |
| MAE | Lowest achieved (~0.214) |
| Stability | Strong across time split |
| Interpretability | High (tree-based + FSI logic) |
| Deployment Ready | Yes (fast, lightweight) |

Why others were rejected:
• Plain Linear → underfitting  
• Neural Networks → unstable on tabular credit data  
• Segmented High Risk → severe overfitting (data sparse)  
• Pure segmentation → inconsistent generalization  

* Final Conclusion:
- This model reaches the **economic ceiling** of borrower-only data
- while remaining **regulator-safe, stable, and production-ready**.


- Pricing Architecture:
-  Core Predictor → LightGBM (log interest rate)
-  Feature Set → Borrower features + FSI + safe interactions
-  Risk Handling → FSI used as explanatory + adjustment signal
-  Output → Stress-adjusted interest rate


In [94]:
import joblib
import lightgbm as lgb

print(type(lgb_model_strong))

joblib.dump(lgb_model_strong, "../models/risk_pricing_lgbm_model.pkl")

<class 'lightgbm.sklearn.LGBMRegressor'>


['../models/risk_pricing_lgbm_model.pkl']

In [92]:
import joblib
import json

# Paths (absolute Windows paths)
MODEL_PATH = r"C:\Users\ASUS\Desktop\EXTRA PROJECT\Financial-Stress-Credit-Pricing-System\models\risk_pricing_lgbm_model.pkl"
SCALER_PATH = r"C:\Users\ASUS\Desktop\EXTRA PROJECT\Financial-Stress-Credit-Pricing-System\models\feature_scaler.pkl"
METADATA_PATH = r"C:\Users\ASUS\Desktop\EXTRA PROJECT\Financial-Stress-Credit-Pricing-System\models\model_metadata.json"

# Example objects (replace with your trained model, scaler, metadata dict)
trained_model = "your_model_object_here"
feature_scaler = "your_scaler_object_here"
metadata = {
    "model_type": "LightGBM",
    "version": "1.0",
    "features": ["age", "income", "loan_amount"]
}

# --- Save model ---
joblib.dump(trained_model, MODEL_PATH)

# --- Save scaler ---
joblib.dump(feature_scaler, SCALER_PATH)

# --- Save metadata ---
with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=4)

print("Model, Scaler, Metadata saved successfully!")

Model, Scaler, Metadata saved successfully!
